# Siklus 2 ER-CyRIS  Uji akhir holdout file 2 dengan aturan v2 dibekukan

**Pakai hanya sekali** setelah kontrak `Kontrak_Klaim_dan_Protokol_Uji_Akhir_UNSW_File2_ER_CyRIS.docx` diperiksa. Notebook ini pertama-tama mereproduksi semua pagar file 1 dan memeriksa hash hasil v2, M2, dan M3. Hanya setelah lolos, notebook menghitung hash file 2, membaca 4.500 baris holdout yang dipilih dalam manifest, dan menguji XGBoost serta Random Forest tanpa belajar ulang dari file 2.

Aturan utama: margin saja dibandingkan dengan gerbang M2+M3 v2 pada kandidat alert yang sama; τd/τx dan ambang detektor lama tetap. `q_e` hanya kolom bukti kelengkapan, bukan veto. Pengukuran gangguan sintetis M2/M3 tidak masuk keputusan. Output: JSON ringkasan dan CSV gzip **baru**; input dan arsip lama tidak ditimpa.

Penafsiran file 2 dibatasi pada transfer antarberkas UNSW. Rentang waktu dapat tumpang tindih dan independensi host tidak dibuktikan. Pekerjaan ini tidak menguji M5–M6, kebenaran risiko/penanganan, atau keunggulan SHAP bila keputusan sama seperti margin.

**Tempatkan berkas Drive:** `UNSW-NB15_1.csv` dan `UNSW-NB15_2.csv` pada `ercyris_siklus2/`; lock dan manifest pada `unsw_locked_protocol_v1/`; JSON v2 pada `unsw_framework_dev_v2/`; dua hasil M2 pada `unsw_m2_stability_file1_v1/`; dua hasil M3 pada `unsw_m3_explanation_independent_file1_v1/`. Ubah hanya variabel folder input bila lokasi Drive Anda berbeda; mengubah aturan numerik membatalkan status uji akhir.


In [ ]:
import csv, hashlib, json, platform, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')
P1=BASE/'UNSW-NB15_1.csv'
P2=BASE/'UNSW-NB15_2.csv'
LOCK_DIR=BASE/'unsw_locked_protocol_v1'
LOCK=LOCK_DIR/'unsw_lock_summary.json'
MANIFEST=LOCK_DIR/'unsw_locked_split_manifest.csv'
V2FILE=BASE/'unsw_framework_dev_v2'/'unsw_framework_development_v2.json'
M2DIR=BASE/'unsw_m2_stability_file1_v1'
M2SUM=M2DIR/'unsw_m2_stability_summary.json'
M2CSV=M2DIR/'unsw_m2_stability_instances.csv.gz'
M3DIR=BASE/'unsw_m3_explanation_independent_file1_v1'
M3SUM=M3DIR/'unsw_m3_independent_summary.json'
M3CSV=M3DIR/'unsw_m3_independent_instances.csv.gz'
OUT=BASE/'unsw_file2_holdout_final_frozen_v1'
OUTFILE=OUT/'unsw_file2_holdout_final_summary.json'
EVIDFILE=OUT/'unsw_file2_holdout_final_instances.csv.gz'
SEED=42; MISSING_RATE=.10; PROBE_SIGMA=.01; SHAP_MAX=300; TOPK=10
TAU_Q=.95; TAU_Q_SEVERE=.70; MIN_GATE_COVERAGE=.50
for path in (P1,P2,LOCK,MANIFEST,V2FILE,M2SUM,M2CSV,M3SUM,M3CSV):
    if not path.is_file(): raise FileNotFoundError(f'Dibutuhkan: {path}')
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Keluaran final sudah ada; STOP agar tidak menimpa.')
print('Output:',OUTFILE,'| File2 belum dibaca; tunggu pemeriksaan hash sebelum sel berikutnya.')

## 1  Rekonstruksi file 1 dan split terkunci

Sampel file 2 masih belum dibaca. Periksa hash, indeks, partisi, dan label dari file 1 seperti v2.


In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE={'label','attack_cat','srcip','dstip','stime','ltime'}
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service'];NUMS=[c for c in FEATURES if c not in CATS]
assert len(FEATURES)==43

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for piece in iter(lambda:f.read(4*1024*1024),b''):h.update(piece)
    return h.hexdigest()

lock=json.loads(LOCK.read_text(encoding='utf-8'))
if lock['status']!='FROZEN_SPLIT_NO_MODEL_NO_HOLDOUT_EVALUATION':
    raise ValueError('Status lock tidak sesuai; STOP.')
if sha256(P1)!=lock['sources']['file1']['sha256'] or sha256(MANIFEST)!=lock['manifest']['sha256']:
    raise ValueError('Berkas sumber atau manifest berubah sejak lock; STOP.')
manifest=pd.read_csv(MANIFEST,dtype={'feature_group_hash64':str})
if len(manifest)!=lock['manifest']['row_count']:
    raise ValueError('Jumlah baris manifest berubah; STOP.')
local=manifest.loc[manifest.source_file==P1.name].copy()
if set(local.partition)!={'train','calibration','development'} or len(local)!=30000:
    raise ValueError('Partisi file1 salah; STOP.')
if manifest.loc[manifest.partition=='holdout','source_file'].nunique()!=1:
    raise ValueError('Holdout bukan tepat satu file; STOP.')
if local.source_row_zero_based.duplicated().any():
    raise ValueError('Nomor baris file1 duplikat lintas partisi; STOP.')
with P1.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
    r=next(csv.reader(f))
if len(r)!=49 or r[-1].strip() not in ('0','1'):
    raise ValueError('Skema sumber tidak cocok; STOP.')
needed=set(local.source_row_zero_based.astype(int)); chunks=[]; offset=0
for chunk in pd.read_csv(P1,header=None,names=COLS,dtype=str,na_filter=False,chunksize=100000):
    positions=np.arange(offset,offset+len(chunk),dtype=np.int32)
    yes=np.isin(positions,list(needed))
    if yes.any():
        z=chunk.loc[yes].copy();z.index=positions[yes];chunks.append(z)
    offset+=len(chunk)
if offset!=lock['sources']['file1']['n']:raise ValueError('Jumlah baris file1 berbeda.')
raw=pd.concat(chunks).sort_index()
if len(raw)!=len(local):raise ValueError('Baris dalam manifest tidak seluruhnya ditemukan.')
labels=pd.to_numeric(raw['label'],errors='coerce')
if labels.isna().any() or not labels.isin([0,1]).all():raise ValueError('Label file1 tidak biner.')
lookup=local.set_index('source_row_zero_based').sort_index()
if not np.array_equal(labels.to_numpy(dtype=int),lookup['label'].to_numpy(dtype=int)):
    raise ValueError('Label manifest tidak sama dengan label sumber.')
for k in ('train','calibration','development'):
    print(k,lock['partitions'][k]['n'],'positif',lock['partitions'][k]['positives'])
print('Manifest tervalidasi. File2 belum dibaca.')

## 2  Praproses dan training train only pada file 1

Setiap parameter transformasi dipelajari hanya dari train file 1. Baris file 2 belum masuk ke model ataupun pemilihan ambang.


In [ ]:
def prepare(frame):
    x=frame[FEATURES].copy()
    for c in NUMS:x[c]=pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in CATS:x[c]=x[c].astype(str).replace('', '__MISSING__')
    return x

def evidence_completeness(x):
    present=x[FEATURES].notna().copy()
    for c in CATS:present[c]&=x[c].astype(str).ne('__MISSING__')
    return present.mean(axis=1).to_numpy(dtype=float)

def synthetic_missing(frame, seed, frac=MISSING_RATE):
    v=frame.copy();rng=np.random.default_rng(seed)
    mask=rng.random((len(v),len(NUMS)))<frac
    for k,c in enumerate(NUMS):v.loc[mask[:,k],c]=np.nan
    return v

parts={name:lookup.loc[lookup.partition==name].index.to_numpy(dtype=np.int32)
       for name in ('train','calibration','development')}
xraw={name:prepare(raw.loc[ix]) for name,ix in parts.items()}
y={name:labels.loc[ix].to_numpy(dtype=np.int8) for name,ix in parts.items()}
xraw['cal_missing']=synthetic_missing(xraw['calibration'],SEED+1)
xraw['dev_missing']=synthetic_missing(xraw['development'],SEED+2)
y['cal_missing']=y['calibration'];y['dev_missing']=y['development']
q={name:evidence_completeness(xraw[name]) for name in xraw}
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
except TypeError:
    enc=OneHotEncoder(handle_unknown='ignore',sparse=False,dtype=np.float32)
prep=ColumnTransformer([
    ('numeric',Pipeline([('median',SimpleImputer(strategy='median')),
                         ('scale',MinMaxScaler())]),NUMS),
    ('categorical',enc,CATS)
],remainder='drop',sparse_threshold=0)
X={'train':np.asarray(prep.fit_transform(xraw['train']),dtype=np.float32)}
for name in xraw:
    if name!='train':X[name]=np.asarray(prep.transform(xraw[name]),dtype=np.float32)
if any(not np.isfinite(v).all() for v in X.values()):raise ValueError('Vektor fitur nonhingga.')
if len({v.shape[1] for v in X.values()})!=1:raise ValueError('Dimensi antarpartisi berbeda.')
print('Fitur hasil train-only transform:',X['train'].shape[1])
print('q_e completeness clean calibration:',float(q['calibration'].mean()),
      'missingness synthetic:',float(q['cal_missing'].mean()))

## 3  Fungsi pengukuran, dengan aturan tetap

Skor M2 dan penjelasan independen dicatat sebagai diagnostik. Keputusan M4 tetap menggunakan margin dan `r_x` dari definisi v2. Fungsi Jaccard hanya untuk memeriksa kesamaan fitur SHAP pada alert yang prediksinya bertahan.


In [ ]:
def select_detector_threshold(y_true,p):
    candidates=np.linspace(.05,.95,19)
    scored=[]
    for t in candidates:
        pred=(p>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
        f=f1_score(y_true,pred,zero_division=0)
        far=float(fp/(fp+tn)) if fp+tn else 1.
        scored.append((float(f),-far,-abs(float(t)-.5),float(t)))
    return max(scored)[3]

def margin_d(probability,detector_threshold):
    p=np.clip(np.asarray(probability,dtype=float),0.,1.)
    return np.clip((p-detector_threshold)/max(1.-detector_threshold,1e-12),0.,1.)

def stable_seed(*labels):
    payload='|'.join(map(str,(SEED,'M3_INDEPENDENT_V1')+labels)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:8],'little')

def jitter_numeric_without_clipping(xmat,seed,sigma=.01):
    out=np.asarray(xmat,dtype=np.float32).copy()
    noise=np.random.default_rng(seed).normal(0,sigma,(len(out),len(NUMS))).astype(np.float32)
    out[:,:len(NUMS)]+=noise
    return out

def tree_shap_positive(explainer,x):
    try:v=explainer.shap_values(x,check_additivity=False)
    except TypeError:v=explainer.shap_values(x)
    if isinstance(v,list):v=v[1]
    v=np.asarray(v,dtype=float)
    if v.ndim==3:
        if v.shape[-1]!=2:raise ValueError('Bentuk SHAP biner tidak sesuai; STOP.')
        v=v[:,:,1]
    if v.shape!=x.shape or not np.isfinite(v).all():raise ValueError('SHAP tidak valid; STOP.')
    return v

def topk_jaccard(base,probe,k=10):
    if base.shape!=probe.shape or base.ndim!=2:raise ValueError('Matriks SHAP tidak kompatibel.')
    k=min(k,base.shape[1]);a=np.argsort(-np.abs(base),axis=1)[:,:k]
    b=np.argsort(-np.abs(probe),axis=1)[:,:k]
    return np.array([len(set(x)&set(y))/len(set(x)|set(y)) for x,y in zip(a,b)],dtype=float)

def gate2(margin,shap_score,tau_d,tau_x):
    a=np.asarray(margin)>=tau_d;b=np.asarray(shap_score)>=tau_x
    return np.where(a&b,'accept',np.where(a|b,'warn','abstain'))


## 4  Buktikan model dan ambang masih mereproduksi v2 pada file 1

Sel berikutnya tidak menyentuh file 2. Jika satu metrik v2 berubah, notebook berhenti.


In [ ]:
models={
 'XGBoost':XGBClassifier(random_state=SEED,n_jobs=-1,eval_metric='logloss',verbosity=0,tree_method='hist'),
 'RandomForest':RandomForestClassifier(random_state=SEED,n_jobs=-1),
}
trained={};thresholds={};probs={};detector_scores={}
for name,model in models.items():
    model.fit(X['train'],y['train']);trained[name]=model
    pcal=model.predict_proba(X['calibration'])[:,1]
    threshold=select_detector_threshold(y['calibration'],pcal);thresholds[name]=threshold
    probs[name]={'calibration':pcal}
    detector_scores[name]={}
    for split in ('calibration','development','cal_missing','dev_missing'):
        prob=pcal if split=='calibration' else model.predict_proba(X[split])[:,1]
        probs[name][split]=prob
        pred=prob>=threshold
        tn,fp,fn,tp=confusion_matrix(y[split],pred,labels=[0,1]).ravel()
        detector_scores[name][split]={
            'n':int(len(pred)),'threshold':threshold,
            'F1':float(f1_score(y[split],pred,zero_division=0)),
            'AP':float(average_precision_score(y[split],prob)),
            'FAR':float(fp/(tn+fp)) if tn+fp else None,
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        }
    print(name,'detector threshold calibration=',threshold,
          'F1 clean development=',round(detector_scores[name]['development']['F1'],4))
reference=json.loads(V2FILE.read_text(encoding='utf-8'))
if reference['status']!='DEVELOPMENT_ONLY_CORRECTED_V2_FILE2_UNTOUCHED':
    raise ValueError('JSON acuan bukan hasil v2 yang diharapkan; STOP.')
if reference['lock_manifest_sha256']!=lock['manifest']['sha256'] or reference['source_file1_sha256']!=lock['sources']['file1']['sha256']:
    raise ValueError('Protokol sumber/manifest berbeda dari v2; STOP.')
for name in trained:
    if abs(thresholds[name]-reference['detector_thresholds_from_calibration'][name])>1e-12:
        raise ValueError(f'Ambang detektor {name} berubah; STOP.')
    for split in ('calibration','development','cal_missing','dev_missing'):
        current=detector_scores[name][split]; earlier=reference['detector_results'][name][split]
        for key in ('n','TN','FP','FN','TP'):
            if current[key]!=earlier[key]:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
        for key in ('F1','AP','FAR'):
            if abs(current[key]-earlier[key])>1e-10:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
print('Rekonstruksi hasil detektor v2 lolos; file2 belum dibaca.')


## 5  Pemeriksaan kontrak sebelum membuka file 2

Nilai SHA berikut diturunkan dari hasil file 1 yang sudah diaudit dan tidak dipilih ulang setelah melihat file 2. Pemeriksaan output sebelumnya dan ambang v2 harus lolos **sebelum** ada operasi baca atas berkas file 2. Pemeriksaan adanya path pada sel pertama bukan pembacaan isi.


In [ ]:
FROZEN={
 'file1':'7d851bbeabd27894ce39c8e78835c73341fc946652fb7743b9eff193b55eb511',
 'manifest':'3cf3c6254b0e0fc356c73516f1a4c0ae009086eeb28c386b0ef00a5a49e532fe',
 'v2':'2ebdc9eba62abdd9308b8bb80893836ac3ec1c36a7f9d05b73866db6b2ba4004',
 'm2summary':'4126fefdf6645952e421850141765db33e6b27073d635b46949ee78b0fd2e959',
 'm2csv':'baf524d4ce2188a01db22950f833f86dcd8a275d667cbce236b8f2ad47ec6ff8',
 'm3summary':'9919dc6ee08a7c834dbaf84efc4aa022705f4e25e3ce8f6d89c1b4e7427b0006',
 'm3csv':'956d9707b7a21a7d028efece7faf2dfe6334a99ae9eca1152150aaa04ca5a006',
}
for label,path in [('file1',P1),('manifest',MANIFEST),('v2',V2FILE),('m2summary',M2SUM),
                   ('m2csv',M2CSV),('m3summary',M3SUM),('m3csv',M3CSV)]:
    if sha256(path)!=FROZEN[label]:raise ValueError('Arsip '+label+' tidak cocok dengan kontrak; STOP.')
prior_m2=json.loads(M2SUM.read_text(encoding='utf-8'))
prior_m3=json.loads(M3SUM.read_text(encoding='utf-8'))
if prior_m2['per_instance_csv_sha256']!=FROZEN['m2csv'] or prior_m3['per_instance_csv_sha256']!=FROZEN['m3csv']:
    raise ValueError('Arsip per alert tidak identik; STOP.')
if prior_m3['m2_summary_sha256']!=FROZEN['m2summary'] or prior_m3['m2_instances_sha256']!=FROZEN['m2csv']:
    raise ValueError('Jejak M3 ke M2 berbeda; STOP.')
if reference['lock_manifest_sha256']!=FROZEN['manifest'] or reference['source_file1_sha256']!=FROZEN['file1']:
    raise ValueError('Jejak v2 ke sumber berbeda; STOP.')
EXPECTED_GATE={'XGBoost':{'tau_d':.9970916956663132,'tau_x':.3855143586678332},
               'RandomForest':{'tau_d':.8499999999999999,'tau_x':.5280114576962448}}
EXPECTED_DETECTOR={'XGBoost':.2,'RandomForest':.6}
for model in EXPECTED_GATE:
    if abs(thresholds[model]-EXPECTED_DETECTOR[model])>1e-12:raise ValueError('Ambang detektor berubah; STOP.')
    for field,v in EXPECTED_GATE[model].items():
        if abs(reference['gate_thresholds_from_calibration'][model][field]-v)>1e-12:
            raise ValueError('Ambang M4 berubah; STOP.')
assert lock['settings']['file2_filter']=='exclude groups present in all file1 rows; deduplicate remaining file2 feature groups'
if lock['partitions']['holdout']['n']!=4500 or manifest.loc[manifest.partition=='holdout','source_file'].nunique()!=1:
    raise ValueError('Jumlah atau sumber holdout berubah; STOP.')
assert reference['config']['holdout_accessed'] is False
assert prior_m2['config']['holdout_accessed'] is False
assert prior_m3['config']['file2_accessed'] is False
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Output final sudah ada; STOP.')
print('SEMUA PAGAR FILE 1 LOLOS. Sel berikutnya pertama kali membuka file 2.')


## 6  Buka file 2 satu kali dan verifikasi indeks holdout

File 2 di-hash seluruhnya lalu dibaca secara streaming; hanya baris 4.500 holdout terpilih yang masuk evaluasi. Label divalidasi terhadap manifest. Kondisi gangguan tambahan 10% memakai benih 1043 (=42+1001), tidak dipilih menurut hasil uji.


In [ ]:
source2sha=sha256(P2)
if P2.name!=lock['sources']['file2']['filename'] or source2sha!=lock['sources']['file2']['sha256']:
    raise ValueError('Checksum file 2 tidak cocok dengan lock; STOP.')
hold=manifest[(manifest.partition=='holdout') & (manifest.source_file==P2.name)].copy()
if len(hold)!=4500 or hold.source_row_zero_based.duplicated().any():
    raise ValueError('Indeks holdout tidak sesuai; STOP.')
wanted=set(hold.source_row_zero_based.astype(int))
pieces=[];total=0
for chunk in pd.read_csv(P2,header=None,names=COLS,dtype=str,na_filter=False,chunksize=100000):
    inds=np.arange(total,total+len(chunk),dtype=np.int32)
    keep=np.isin(inds,list(wanted))
    if keep.any():
        part=chunk.loc[keep].copy();part.index=inds[keep];pieces.append(part)
    total+=len(chunk)
if total!=lock['sources']['file2']['n']:
    raise ValueError('Jumlah baris sumber file 2 berubah; STOP.')
holdraw=pd.concat(pieces).sort_index()
hold=hold.set_index('source_row_zero_based').sort_index()
if len(holdraw)!=len(hold) or not np.array_equal(holdraw.index.to_numpy(),hold.index.to_numpy()):
    raise ValueError('Indeks holdout yang terbaca tidak sesuai; STOP.')
holdlabels=pd.to_numeric(holdraw.label,errors='coerce')
if holdlabels.isna().any() or not np.array_equal(holdlabels.to_numpy(int),hold.label.to_numpy(int)):
    raise ValueError('Label sumber berbeda dari label holdout; STOP.')
xraw_h=prepare(holdraw)
xraw_h_missing=synthetic_missing(xraw_h,SEED+1001,frac=.10)
Xh=np.asarray(prep.transform(xraw_h),dtype=np.float32)
Xhm=np.asarray(prep.transform(xraw_h_missing),dtype=np.float32)
yh=holdlabels.to_numpy(dtype=np.int8)
source_rows=holdraw.index.to_numpy(dtype=int)
q_h=evidence_completeness(xraw_h)
q_hm=evidence_completeness(xraw_h_missing)
if len(Xh)!=4500 or Xh.shape[1]!=X['train'].shape[1] or not np.isfinite(Xh).all() or not np.isfinite(Xhm).all():
    raise ValueError('Transformasi holdout tidak valid; STOP.')
print('SUMBER FILE 2 DIBUKA: 4.500 baris holdout. Tidak ada perubahan model ataupun ambang.')


## 7  Ukur kinerja, perbandingan gerbang, dan diagnostik tambahan

Kode ini tidak mengoptimasi aturan. Skor `r_x` asli direproduksi dengan probe v2 ber-clipping; M3 independent Jaccard menggunakan dua benih lain tanpa clipping. M2 memakai dua keluarga gangguan (masing-masing 12 pengulangan), hanya untuk diagnostik stabilitas. Penjelasan dihitung untuk **semua** kandidat alert tanpa sampling; SHAP diproses dalam batch agar batas 300 kandidat dari v2 tidak mengecualikan alert file 2. Waktu hanya untuk komponen evaluasi di sel ini, bukan klaim latensi end-to-end antar-siklus.


In [ ]:
def add_numeric_probe_v2(xmat):
    v=np.asarray(xmat,dtype=np.float32).copy()
    rng=np.random.default_rng(SEED+5)
    v[:,:len(NUMS)]=np.clip(v[:,:len(NUMS)]+rng.normal(0,.01,(len(v),len(NUMS))),0,1)
    return v

def shap_stability_v2(base,probe,k=TOPK):
    a=np.asarray(base,dtype=float);b=np.asarray(probe,dtype=float)
    if a.shape!=b.shape or a.ndim!=2:raise ValueError('Matriks SHAP tidak cocok; STOP.')
    k=min(k,a.shape[1]);aa=np.abs(a);bb=np.abs(b)
    ta=np.argsort(-aa,axis=1)[:,:k];tb=np.argsort(-bb,axis=1)[:,:k]
    jac=[];sign=[]
    for i,(left,right) in enumerate(zip(ta,tb)):
        union=set(left)|set(right)
        jac.append(len(set(left)&set(right))/len(union))
        pos=sorted(union);sign.append(float(np.mean(np.sign(a[i,pos])==np.sign(b[i,pos]))))
    den=aa.mean(axis=1)+1e-12
    mag=np.clip(1-np.abs(aa-bb).mean(axis=1)/den,0,1)
    return np.clip(np.column_stack([jac,mag,sign]),0,1).mean(axis=1)

def batched_shap(explainer,mat,batch=128):
    if len(mat)==0:return np.empty((0,mat.shape[1]))
    return np.concatenate([tree_shap_positive(explainer,mat[i:i+batch])
                           for i in range(0,len(mat),batch)],axis=0)

def count_gate(action,truth):
    accept=action=='accept'
    return {'candidate_alerts':int(len(action)),'accept':int(accept.sum()),
            'warn':int((action=='warn').sum()),'abstain':int((action=='abstain').sum()),
            'TP_accept':int((accept & (truth==1)).sum()),'FP_accept':int((accept & (truth==0)).sum()),
            'accept_coverage':float(accept.mean()) if len(action) else None,
            'true_alerts_not_accepted':int(((~accept) & (truth==1)).sum())}

NOISE_PROBES=12;MISSING_PROBES=12;NUM_NEW_SHAP_PROBES=2
records=[];results={};evaluation_timings={}
for name,model in trained.items():
    results[name]={};evaluation_timings[name]={}
    explainer=shap.TreeExplainer(model)
    dt=thresholds[name];gate=reference['gate_thresholds_from_calibration'][name]
    for condition,xmat,xframe,quality in [('clean',Xh,xraw_h,q_h),
                                          ('synthetic_missing',Xhm,xraw_h_missing,q_hm)]:
        start=time.perf_counter()
        prob=model.predict_proba(xmat)[:,1]
        pred=prob>=dt
        tn,fp,fn,tp=confusion_matrix(yh,pred,labels=[0,1]).ravel()
        detector={'n':int(len(yh)),'positive_count':int(yh.sum()),'TP':int(tp),'FP':int(fp),
                  'FN':int(fn),'TN':int(tn),
                  'F1':float(f1_score(yh,pred,zero_division=0)),
                  'AP':float(average_precision_score(yh,prob)),
                  'FAR':float(fp/(fp+tn)) if fp+tn else None}
        idx=np.flatnonzero(pred)
        cand=xmat[idx];rawcand=xframe.iloc[idx]
        base=batched_shap(explainer,cand)
        clipped=batched_shap(explainer,add_numeric_probe_v2(cand))
        rx=shap_stability_v2(base,clipped)
        md=margin_d(prob[idx],dt)
        original_action=gate2(md,rx,gate['tau_d'],gate['tau_x'])
        margin_accept=md>=gate['tau_d']
        truth=yh[idx]
        margin_action=np.where(margin_accept,'accept','warn')
        gate_stats=count_gate(original_action,truth)
        margin_stats=count_gate(margin_action,truth)
        if not np.all(original_action[margin_accept] == np.where(rx[margin_accept]>=gate['tau_x'],'accept','warn')):
            raise ValueError('Gerbang M4 dan margin tidak konsisten; STOP.')
        noise_hits=np.zeros(len(idx),dtype=float)
        missing_hits=np.zeros(len(idx),dtype=float)
        for rep in range(NOISE_PROBES):
            altered=jitter_numeric_without_clipping(cand,stable_seed('M2_FILE2',name,condition,'noise',rep),.01)
            noise_hits+=model.predict_proba(altered)[:,1]>=dt
        for rep in range(MISSING_PROBES):
            alter_raw=synthetic_missing(rawcand,stable_seed('M2_FILE2',name,condition,'missing',rep),frac=.05)
            altered=np.asarray(prep.transform(alter_raw),dtype=np.float32)
            missing_hits+=model.predict_proba(altered)[:,1]>=dt
        jaccard=[];retention=[]
        for rep in range(NUM_NEW_SHAP_PROBES):
            altered=jitter_numeric_without_clipping(cand,stable_seed(name,'holdout',condition,'heldout_shap',rep),.01)
            jaccard.append(topk_jaccard(base,batched_shap(explainer,altered),TOPK))
            retention.append(model.predict_proba(altered)[:,1]>=dt)
        jaccard=np.column_stack(jaccard);retention=np.column_stack(retention)
        both_retained=retention.all(axis=1)
        jmean=jaccard.mean(axis=1)
        a=original_action=='accept'
        m=margin_accept
        result={'detector':detector,'margin_only':margin_stats,'two_domain_gate':gate_stats,
                'q_e_completeness_mean_on_alerts':float(quality[idx].mean()) if len(idx) else None,
                'M2_mean_s_d_noise':float((noise_hits/NOISE_PROBES).mean()) if len(idx) else None,
                'M2_mean_s_d_missing':float((missing_hits/MISSING_PROBES).mean()) if len(idx) else None,
                'candidate_alerts_retain_both_new_probes':int(both_retained.sum()),
                'margin_accept_retain_both_new_probes':int((m&both_retained).sum()),
                'gate_accept_retain_both_new_probes':int((a&both_retained).sum()),
                'SHAP_only_exclusions_after_margin':int((m&~a).sum()),
                'SHAP_only_excluded_TP':int((m&~a&(truth==1)).sum()),
                'SHAP_only_excluded_FP':int((m&~a&(truth==0)).sum()),
                'mean_jaccard_given_stable_prediction':float(jmean[both_retained].mean()) if both_retained.any() else None}
        results[name][condition]=result
        records.append(pd.DataFrame({'source_file':P2.name,'source_row_zero_based':source_rows[idx],
             'model':name,'condition':condition,'y_true':truth,'probability':prob[idx],
             'detector_threshold':dt,'m_d_margin':md,'r_x_local_shap':rx,
             'q_completeness':quality[idx],'margin_only_accept':m.astype(int),
             'two_domain_action':original_action,'s_d_noise':noise_hits/NOISE_PROBES,
             's_d_missing':missing_hits/MISSING_PROBES,
             'probe_1_jaccard':jaccard[:,0],'probe_2_jaccard':jaccard[:,1],
             'both_independent_probes_alert_retained':both_retained.astype(int),
             'mean_independent_jaccard':jmean}))
        evaluation_timings[name][condition]=float(time.perf_counter()-start)
        print(name,condition,'TP/FP detector',tp,fp,'accept margin',margin_stats['accept'],
              'accept M4',gate_stats['accept'],'SHAP-specific exclusions',result['SHAP_only_exclusions_after_margin'])

instances=pd.concat(records,ignore_index=True)
if instances.duplicated(['model','condition','source_row_zero_based']).any():
    raise ValueError('Identitas hasil per alert duplikat; STOP.')


## 8  Simpan dua hasil kecil dan beri batas klaim

Nol false accept pada sampel tidak sama dengan jaminan sistem. Bila margin saja dan M4 sama, keberhasilan memilih accept tidak boleh dikaitkan dengan SHAP. Angka waktu di bawah ini hanya durasi komponen evaluasi yang dieksekusi pada runtime sekarang; jangan membandingkannya langsung dengan siklus lain tanpa kesetaraan lingkungan. Unggah hanya JSON dan CSV gzip kecil, bukan data mentah UNSW.


In [ ]:
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Output final sudah ada; STOP.')
OUT.mkdir(parents=True,exist_ok=True)
instances.to_csv(EVIDFILE,index=False,compression={'method':'gzip','mtime':0})
audit={'status':'UNSW_FILE2_FROZEN_FINAL_NO_TUNING',
       'reference_sha256':FROZEN,'source_file2_sha256':source2sha,
       'lock_protocol_id':lock['protocol_id'],'lock_manifest_sha256':sha256(MANIFEST),
       'source_file2_rows':int(total),'selected_holdout_rows':int(len(yh)),
       'holdout_context_audit':lock['context_audit'],
       'config':{'train_seed':SEED,'synthetic_missing_fraction':.10,
                 'synthetic_missing_seed':SEED+1001,
                 'detector_thresholds':EXPECTED_DETECTOR,
                 'gate_thresholds':EXPECTED_GATE,
                 'quality_veto':False,'M2_probe_count_per_family':12,
                 'independent_M3_probe_count':NUM_NEW_SHAP_PROBES,
                 'source2_used_for_training_or_thresholds':False},
       'metrics':results,'per_instance_count':int(len(instances)),
       'per_instance_csv_sha256':sha256(EVIDFILE),
       'evaluation_seconds_only':evaluation_timings,
       'runtime':{'python':platform.python_version(),'pandas':pd.__version__,
                  'numpy':np.__version__,'sklearn':__import__('sklearn').__version__,
                  'xgboost':__import__('xgboost').__version__,'shap':shap.__version__},
       'limitations':['Holdout measures limited file transfer, not independent host or time deployment',
                      'Clean and 10 percent synthetic-missing outcomes share the same source rows',
                      'M2 robustness scores are diagnostics, not calibrated correctness',
                      'M3 Jaccard is a synthetic consistency measure on transformed feature indices, not explanation fidelity or raw-field meaning',
                      'No provenance validation for M1, no organizational risk/response validation M5/M6',
                      'Any change proposed after this file2 result needs a genuinely new test set']}
OUTFILE.write_text(json.dumps(audit,indent=2,ensure_ascii=False),encoding='utf-8')
print('Keluaran akhir:',OUTFILE,EVIDFILE)
